In [ ]:
# Step 1 Install dependencies [For environment setup: Install required Python packages]
!pip install ultralytics

In [ ]:
# Step 2 Mount Google Drive [For environment setup: Access files from Google Drive]
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3 Config + imports [For data processing: Configure dataset and training paths]
from pathlib import Path
import csv
import shutil
from ultralytics import YOLO

DATA_YAML = Path('/content/drive/MyDrive/BadmintonEditor/data/shuttle_dataset_test/data.yaml')
BASE_MODEL = 'yolov8n.pt'
EPOCHS = 50
IMGSZ = 640
OUTPUT_MODEL = Path('/content/drive/MyDrive/BadmintonEditor/trained_models/shuttle_detector/shuttlecock_yolo.pt')

In [ ]:
# Step 4 Core helper functions [For data processing: Parse YAML and map image-label paths]
def print_section(title: str) -> None:
    line = '=' * 30
    print(f'\n{line}\n{title}\n{line}')


def parse_simple_yaml(yaml_path: Path) -> dict[str, str]:
    data: dict[str, str] = {}
    for raw in yaml_path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        k, v = line.split(':', 1)
        data[k.strip()] = v.strip()
    return data


def resolve_dataset_root(yaml_path: Path, root_raw: str) -> Path:
    candidate = Path(root_raw).expanduser()
    if candidate.is_absolute():
        return candidate.resolve()
    cwd_resolved = candidate.resolve()
    if cwd_resolved.exists():
        return cwd_resolved
    return (yaml_path.parent / candidate).resolve()


def iter_split_images(dataset_root: Path, split_value: str) -> list[Path]:
    split_path = Path(split_value).expanduser()
    if not split_path.is_absolute():
        split_path = (dataset_root / split_path).resolve()

    if split_path.is_file():
        out = []
        for raw in split_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line:
                continue
            p = Path(line).expanduser()
            if not p.is_absolute():
                p = (dataset_root / p).resolve()
            out.append(p)
        return out

    if split_path.is_dir():
        exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
        return sorted(p for p in split_path.rglob('*') if p.is_file() and p.suffix.lower() in exts)

    return []


def infer_label_path(image_path: Path, dataset_root: Path) -> Path:
    s = str(image_path)
    if '/images/' in s:
        return Path(s.replace('/images/', '/labels/')).with_suffix('.txt')
    try:
        rel = image_path.relative_to(dataset_root)
        return (dataset_root / 'labels' / rel.parent / f'{image_path.stem}.txt').resolve()
    except ValueError:
        return image_path.with_suffix('.txt')

In [ ]:
# Step 5 Dataset preprocessing function [For data processing: Create missing empty label files]
def preprocess_missing_empty_labels(data_yaml_path: Path) -> dict[str, int | str]:
    payload = parse_simple_yaml(data_yaml_path)
    root_raw = payload.get('path')
    if not root_raw:
        raise ValueError(f"'path' missing in dataset YAML: {data_yaml_path}")

    dataset_root = resolve_dataset_root(data_yaml_path, root_raw)
    split_keys = [k for k in ('train', 'val', 'test') if payload.get(k)]
    if not split_keys:
        raise ValueError('YAML must define at least one split: train/val/test')

    total_images = 0
    existing_labels = 0
    created_empty = 0

    for split_key in split_keys:
        images = iter_split_images(dataset_root, payload[split_key])
        total_images += len(images)
        for image_path in images:
            label_path = infer_label_path(image_path, dataset_root)
            if label_path.exists():
                existing_labels += 1
            else:
                label_path.parent.mkdir(parents=True, exist_ok=True)
                label_path.write_text('', encoding='utf-8')
                created_empty += 1

    return {
        'dataset_root': str(dataset_root),
        'total_images': total_images,
        'existing_labels': existing_labels,
        'created_empty_labels': created_empty,
    }

In [ ]:
# Step 6 Metrics helper functions [For print: Parse and format compact training metrics]
def count_val_instances(data_yaml_path: Path) -> int:
    payload = parse_simple_yaml(data_yaml_path)
    root_raw = payload.get('path')
    val_raw = payload.get('val')
    if not root_raw or not val_raw:
        return 0
    dataset_root = resolve_dataset_root(data_yaml_path, root_raw)
    val_images = (dataset_root / Path(val_raw)).resolve()
    val_labels = Path(str(val_images).replace('/images/', '/labels/'))
    if not val_labels.exists():
        return 0
    total = 0
    for txt in val_labels.glob('*.txt'):
        total += len([ln for ln in txt.read_text(encoding='utf-8').splitlines() if ln.strip()])
    return total


def to_float(row: dict[str, str], key: str) -> float:
    try:
        return float(row.get(key, '0'))
    except Exception:
        return 0.0


def estimate_tp_fp_fn(precision: float, recall: float, gt_instances: int) -> tuple[int, int, int]:
    tp = max(0.0, recall * gt_instances)
    fn = max(0.0, gt_instances - tp)
    fp = 0.0 if precision <= 0 else max(0.0, tp * (1.0 / precision - 1.0))
    return int(round(tp)), int(round(fp)), int(round(fn))


def print_compact_metrics(results_csv_path: Path, data_yaml_path: Path) -> None:
    if not results_csv_path.exists():
        print(f'[metrics] results.csv not found: {results_csv_path}')
        return

    gt_instances = count_val_instances(data_yaml_path)
    rows = list(csv.DictReader(results_csv_path.open('r', encoding='utf-8', newline='')))
    if not rows:
        print(f'[metrics] no rows in: {results_csv_path}')
        return

    header = 'epoch | box_loss | cls_loss | dfl_loss |   P    |   R    | mAP50  | mAP50-95 | TP  | FP  | FN '
    print('\nCompact metrics per epoch')
    print(header)
    print('-' * len(header))

    for row in rows:
        epoch = int(to_float(row, 'epoch'))
        box_loss = to_float(row, 'train/box_loss')
        cls_loss = to_float(row, 'train/cls_loss')
        dfl_loss = to_float(row, 'train/dfl_loss')
        p = to_float(row, 'metrics/precision(B)')
        r = to_float(row, 'metrics/recall(B)')
        map50 = to_float(row, 'metrics/mAP50(B)')
        map5095 = to_float(row, 'metrics/mAP50-95(B)')
        tp, fp, fn = estimate_tp_fp_fn(p, r, gt_instances)

        print(f'{epoch:>5} | {box_loss:>8.4f} | {cls_loss:>8.4f} | {dfl_loss:>8.4f} | {p:>6.4f} | {r:>6.4f} | {map50:>6.4f} | {map5095:>8.4f} | {tp:>3} | {fp:>3} | {fn:>3}')

    print(f'\n[metrics] TP/FP/FN estimated from P/R with val GT count={gt_instances}.')

In [ ]:
# Step 7 Precheck + run dataset preprocessing [For data processing: Validate YAML and execute preprocessing]
if not DATA_YAML.exists():
    raise FileNotFoundError(f'Not found: {DATA_YAML}')

print_section('DATASET PREPROCESSING')
prep = preprocess_missing_empty_labels(DATA_YAML)
print(f"dataset root: {prep['dataset_root']}")
print(f"total images checked: {prep['total_images']}")
print(f"existing label files: {prep['existing_labels']}")
print(f"new empty labels created: {prep['created_empty_labels']}")

In [ ]:
# Step 8 Train YOLO model [For train: Run model training epochs]
print_section('TRAINING')
model = YOLO(BASE_MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    task='detect',
    verbose=True,
)
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'

In [ ]:
# Step 9 Print compact metrics [For print: Show key training metrics table]
print_section('RESULTS')
print_compact_metrics(Path(results.save_dir) / 'results.csv', DATA_YAML)

In [ ]:
# Step 10 Save best model to Drive [For output: Persist final model artifact]
OUTPUT_MODEL.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best_pt, OUTPUT_MODEL)
print(f'Trained model copied to: {OUTPUT_MODEL}')